In [ ]:
import pandas as pd 
import numpy as np 

dataset = pd.read_csv("../data/processed/smart_waste_dataset_cleaned.csv", parse_dates=["ArrivalDate", "ExpiryDate"])
dataset.shape

(1000, 21)

#### 1. Shelf life features

In [ ]:
# Total days the item was ever given between arrival and expiry

dataset["Total_Shelf_Life_Days"] = (dataset["ExpiryDate"] - dataset["ArrivalDate"]).dt.days
dataset["Days_In_Store_So_Far"] = (dataset["Total_Shelf_Life_Days"] - dataset["DaysUntilExpiry"]).clip(lower=0)

### 2. Sales Velocity vs. stock (the core risk signal)

In [ ]:
dataset["Days_To_Clear_Stock"] = np.where(
    dataset["DailySaleAvg"] > 0, 
    dataset["StockQty"] / dataset["DailySaleAvg"],
    np.inf
 )

dataset["Stock_Coverage_Ratio"] = dataset["Days_To_Clear_Stock"] / dataset["DaysUntilExpiry"].replace(0, np.nan)
dataset["Stock_Coverage_Ratio"] = dataset["Stock_Coverage_Ratio"].fillna(dataset["Stock_Coverage_Ratio"].max())

### 3. Cross-store demand comparison


In [ ]:
dataset["Sales_vs_Nearby_Gap"] = dataset["DailySaleAvg"] - dataset["AvgDailySaleInNearbyStores"]

### 4. Sanity check

In [ ]:
dataset.groupby("IsSpoiled")[["Stock_Coverage_Ratio", "Days_In_Store_So_Far", "Sales_vs_Nearby_Gap"]].mean().round(2)

,Stock_Coverage_Ratio,Days_In_Store_So_Far,Sales_vs_Nearby_Gap
IsSpoiled,,,
False,19.97,5.00,0.10
True,69.39,5.17,-0.07


In [ ]:
# Saving
out_path = "../data/processed/smart_waste_features.csv"
dataset.to_csv(out_path, index=False)
print(f"Saved: {dataset.shape[0]} rows, {dataset.shape[1]} columns -> {out_path}")

Saved: 1000 rows, 26 columns -> ../data/processed/smart_waste_features.csv
